In [7]:
import rdkit.Chem.AllChem as AllChem
from numpy import zeros
from rdkit import DataStructs
from rdkit.Chem import MolFromSmiles
from sklearn.model_selection import train_test_split, GridSearchCV, RepeatedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, balanced_accuracy_score, matthews_corrcoef, f1_score
from pandas import DataFrame, concat
from pickle import load

In [11]:
def calc_morgan(mols):
    """ генерация молекулярных отпечатков по методу Моргана с радиусом 2 и длиной 2048
    """
    for_df = []
    for m in mols:
        arr = zeros((1,), dtype=int)
        DataStructs.ConvertToNumpyArray(AllChem.GetMorganFingerprintAsBitVect(m, 2, 2048), arr)
        for_df.append(arr)
    return DataFrame(for_df)

with open('../data/classifier/modeling_data.pickle', 'rb') as file:
    data = load(file)

In [12]:
molecules = [
    MolFromSmiles(mol) for mol in data['SMILES']
]

In [ ]:
X = calc_morgan(molecules)
Y = data['Activity']
XY = concat((X, Y), axis=1)
XY

[12:02:13] DEPRECATION WARNING: please use MorganGenerator
[12:02:13] DEPRECATION WARNING: please use MorganGenerator
[12:02:13] DEPRECATION WARNING: please use MorganGenerator
[12:02:13] DEPRECATION WARNING: please use MorganGenerator
[12:02:13] DEPRECATION WARNING: please use MorganGenerator
[12:02:13] DEPRECATION WARNING: please use MorganGenerator
[12:02:13] DEPRECATION WARNING: please use MorganGenerator
[12:02:13] DEPRECATION WARNING: please use MorganGenerator
[12:02:13] DEPRECATION WARNING: please use MorganGenerator
[12:02:13] DEPRECATION WARNING: please use MorganGenerator
[12:02:13] DEPRECATION WARNING: please use MorganGenerator
[12:02:13] DEPRECATION WARNING: please use MorganGenerator
[12:02:13] DEPRECATION WARNING: please use MorganGenerator
[12:02:13] DEPRECATION WARNING: please use MorganGenerator
[12:02:13] DEPRECATION WARNING: please use MorganGenerator
[12:02:13] DEPRECATION WARNING: please use MorganGenerator
[12:02:13] DEPRECATION WARNING: please use MorganGenerat

In [16]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.20)

In [ ]:
rf = RandomForestClassifier(class_weight='balanced')
pg = {'n_estimators': [50, 100, 200, 300, 400, 500],
      'max_depth': [None, 1, 3, 5, 7, 10],
      }
cv = RepeatedKFold(n_splits=5, n_repeats=5)
gs = GridSearchCV(rf, pg, verbose=3, cv=cv, refit='f1', scoring=('f1', 'balanced_accuracy'))

gs.fit(x_train, y_train)

Fitting 25 folds for each of 36 candidates, totalling 900 fits
[CV 1/25] END max_depth=None, n_estimators=50; balanced_accuracy: (test=0.721) f1: (test=0.963) total time=  21.3s
[CV 2/25] END max_depth=None, n_estimators=50; balanced_accuracy: (test=0.718) f1: (test=0.963) total time=  21.5s
[CV 3/25] END max_depth=None, n_estimators=50; balanced_accuracy: (test=0.716) f1: (test=0.963) total time=  21.7s
[CV 4/25] END max_depth=None, n_estimators=50; balanced_accuracy: (test=0.711) f1: (test=0.961) total time=  21.3s
[CV 5/25] END max_depth=None, n_estimators=50; balanced_accuracy: (test=0.716) f1: (test=0.962) total time=  21.7s
[CV 6/25] END max_depth=None, n_estimators=50; balanced_accuracy: (test=0.723) f1: (test=0.963) total time=  21.5s
[CV 7/25] END max_depth=None, n_estimators=50; balanced_accuracy: (test=0.721) f1: (test=0.961) total time=  21.6s
[CV 8/25] END max_depth=None, n_estimators=50; balanced_accuracy: (test=0.718) f1: (test=0.962) total time=  21.7s
[CV 9/25] END max

In [20]:
gs.best_estimator_

RandomForestClassifier(class_weight='balanced', n_estimators=500)

In [21]:
y_pred = gs.predict(x_test)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel().tolist()
precision = tp / (tp + fp)
recall = tp / (tp + fn)

In [22]:
print(f'ACC = {accuracy_score(y_test, y_pred):.3f}')
print(f'BA = {balanced_accuracy_score(y_test, y_pred):.3f}')
print(f'MCC = {matthews_corrcoef(y_test, y_pred):.3f}')
print(f'ROC AUC = {roc_auc_score(y_test, y_pred):.3f}')
print(f'F1 = {f1_score(y_test, y_pred):.3f}')
print(f'Precision = {precision:.3f}')
print(f'Recall = {recall:.3f}')

ACC = 0.935
BA = 0.733
MCC = 0.592
ROC AUC = 0.733
F1 = 0.965
Precision = 0.943
Recall = 0.987
